# 13 — Comparación de modelos con las features finales (Paso 7)

Repite la comparación de la sección 3.3 de `train.ipynb` (los 11 modelos del material + 3 ensambles), pero con las **features finales**. Esta tabla es la de la presentación ("selección de modelos").

**Se elige qué modelos correr** en la celda "Qué modelos correr" (abajo): cada modelo tiene un `True`/`False`, y las celdas de los que están en `False` se saltean. La comparación completa de los 14 modelos ya está en la Fase A (`train.ipynb`, con las features de ese momento); acá alcanza con un modelo por familia.

**Qué cambia respecto de `train.ipynb`:**

1. **Features.** Se suman:
   - `prob_texto_new`: `PrediccionOOF(armar_modelo_texto())`, el modelo de texto de título + garantía con *cross-fitting* (`09_texto`, `11b_marginales`).
   - Las 6 columnas de `COLUMNAS_CONTENIDO`: fotos (N) y fechas (L), calculadas con `agregar_features_contenido` (`11_anidadas`, `07_nuevas_features`, `11b_marginales`).
   - `duracion_dias` (`stop_time` − `start_time`, `18_barrido_campos`), que está en los dos modelos del modelo final.
2. **Imputación.** Las features de fotos tienen nulos (0,8% de las filas: publicaciones sin fotos o con un tamaño ilegible). El HistGB los maneja solo, pero regresión logística, LDA, kNN o SVM no aceptan nulos. Uso `SimpleImputer(strategy="median", add_indicator=True)` adentro del `Pipeline`, como en `material/Variables Categóricas, Datos Faltantes y Feature Engineering/Copia de AP_Categóricas.ipynb` (imputación por mediana, recomendada para variables asimétricas, y una variable booleana que marca qué dato faltaba). La mediana se calcula solo con el train de cada fold.
3. **`refit="accuracy"`** (antes `"roc_auc"`): la métrica de la competencia es accuracy.
4. **Caché del preprocesamiento** (`Pipeline(memory=...)`, igual que en `12_tuning`). El preprocesamiento es **el mismo para todos los modelos**, así que el modelo de texto se ajusta una vez por fold y lo reusan todas las combinaciones de todos los modelos.
5. **Comparación pareada** del mejor modelo contra el resto.
6. **Los modelos del modelo final**, cada uno con **su propio preprocesamiento** (el de `features.py`):
   - **CatBoost** (`armar_pipeline_final()`): categóricas crudas, parámetros por defecto (el tuning de `laboratorio.ipynb`, C2, no lo mejoró);
   - **HistGB con combinaciones** (`armar_pipeline_histgb_combos()`): `TargetEncoder` sobre vendedor, categoría y las combinaciones de `agregar_combos`;
   - **Promedio 50/50** de los dos (P7c), que es el **modelo final** (`train.ipynb` / `predict.ipynb`).

   Los 11 modelos de sklearn de la grilla usan la codificación anterior (`VendedorEncoder` + `TargetEncoder` + one-hot); los lineales y kNN no aceptan categóricas crudas. Es comparar cada modelo con la mejor codificación que se probó para él.

Los hiperparámetros y grillas de cada modelo son **los mismos de `train.ipynb`**, con la misma justificación y material (se resume en cada inciso).

**Stacking:** `material/modelos de ensamble/S8_Ensambles.ipynb` aclara que Stacking "no forma parte de la versión actual de la clase". Se mantiene igual (decisión tomada): es el ensamble que mejor dio en `train.ipynb` y usa la misma idea que `PrediccionOOF` (un modelo que aprende sobre predicciones *out-of-fold* de otros).

In [1]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
import warnings
import tempfile
import shutil
import time
from joblib import Memory

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer, StandardScaler, TargetEncoder
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    AdaBoostClassifier,
    HistGradientBoostingClassifier,
    VotingClassifier,
    StackingClassifier,
)
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_validate

from features import (
    aplanar_anidadas, agregar_logs, agregar_features_contenido, agregar_categoricas_crudas, agregar_combos,
    CapeoPorMoneda, VendedorEncoder, PrediccionOOF, armar_modelo_texto, armar_pipeline_final,
    armar_pipeline_histgb_combos,
    COLUMNAS_CONTENIDO, COLUMNAS_CAT_CRUDAS, INDICES_CAT,
)

SEED = 42

train = pd.read_json("../data/train_data.jsonlines", lines=True)
y = (train["condition"] == "new").astype(int)
train = aplanar_anidadas(train)
train = agregar_combos(agregar_categoricas_crudas(agregar_features_contenido(train)))

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

## Qué modelos correr

Poné `True` en los que quieras correr y `False` en los que no. Por defecto, un modelo por familia (lineal, árbol, *bagging*, *boosting*) + el modelo final:
- **Regresión logística** (lineal; además es el baseline), **Árbol de decisión**, **Random Forest** (*bagging*), **Gradient Boosting** (HistGB, *boosting*), y **CatBoost + HistGB con combinaciones + promedio** (el modelo final).
- Apagados por defecto: QDA y Naive Bayes (0,72 y 0,64 en la Fase A), LDA y SVM lineal (≈ regresión logística), kNN (el más lento), Extra Trees y AdaBoost (variantes de *bagging* y *boosting*), Voting y Stacking (`16_ensambles` ya mostró que no suman).

Los ensambles (Voting, Stacking) combinan **solo los modelos que se corrieron**. Necesitan al menos 2.

In [2]:
MODELOS = {
    "Regresión logística": True,
    "LDA": False,
    "QDA": False,
    "Naive Bayes (Gaussiano)": False,
    "kNN": False,
    "SVM lineal": False,
    "Árbol de decisión": False,
    "Random Forest": True,
    "Extra Trees": False,
    "AdaBoost": False,
    "Gradient Boosting": True,
    "Voting (soft)": True,
    "Voting (hard)": False,
    "Stacking": True,
    "Modelo final (CatBoost + HistGB con combinaciones + promedio)": True,
}


def correr(nombre):
    if not MODELOS[nombre]:
        print(f"{nombre}: salteado (False en MODELOS)")
    return MODELOS[nombre]


print("Se corren:", [m for m, si in MODELOS.items() if si])

Se corren: ['Regresión logística', 'Random Forest', 'Gradient Boosting', 'Voting (soft)', 'Stacking', 'Modelo final (CatBoost + HistGB con combinaciones + promedio)']


## Preprocesador compartido

El mismo `preprocesador_comparacion` de `train.ipynb` (One-Hot con `drop="first"`, `StandardScaler` para las numéricas, booleanas sin tocar, TF-IDF de 500, `VendedorEncoder`, `TargetEncoder` de `category_id`) más dos transformers nuevos:

- `contenido`: imputación por mediana + indicador de faltante → escalado, sobre las 6 columnas de fotos y fechas + `duracion_dias`.
- `texto`: `PrediccionOOF(armar_modelo_texto())` sobre `title` y `warranty_texto`.

`train_modelo` deja solo las columnas que usa el modelo, para que la caché compare rápido "¿son las mismas filas?".

In [3]:
columnas_categoricas = ["listing_type_id", "buying_mode", "shipping_mode", "seller_state"]
columnas_escala = [
    "log_price", "log_available_quantity",
    "sold_quantity", "initial_quantity",
    "n_fotos", "n_atributos", "n_variaciones", "n_metodos_pago",
]
columnas_bool = [
    "automatic_relist",
    "tiene_video", "tiene_garantia",
    "free_shipping", "local_pick_up", "accepts_mercadopago",
    "es_tienda_oficial", "tiene_descuento", "tiene_oferta", "tiene_parent_item",
]

columnas_contenido = COLUMNAS_CONTENIDO + ["duracion_dias"]   # fotos, fechas y duración

preprocesador_comparacion = ColumnTransformer([
    ("onehot", OneHotEncoder(handle_unknown="ignore", drop="first"), columnas_categoricas),
    ("escala", StandardScaler(), columnas_escala),
    ("bool", "passthrough", columnas_bool),
    ("contenido", Pipeline([
        ("imputacion", SimpleImputer(strategy="median", add_indicator=True)),
        ("escala", StandardScaler()),
    ]), columnas_contenido),
    ("tfidf", TfidfVectorizer(max_features=500), "title"),
    ("texto", PrediccionOOF(armar_modelo_texto(), random_state=SEED), ["title", "warranty_texto"]),
    ("vendedor", VendedorEncoder(incluir_target=True), ["seller_id", "log_price"]),
    ("categoria", TargetEncoder(target_type="binary", random_state=SEED), ["category_id"]),
], sparse_threshold=0)

columnas_crudas = ["price", "base_price", "currency_id", "available_quantity", "title", "warranty_texto",
                   "seller_id", "category_id"] + COLUMNAS_CAT_CRUDAS
columnas_modelo = [c for c in dict.fromkeys(columnas_categoricas + columnas_escala + columnas_bool
                                            + columnas_contenido + columnas_crudas)
                   if c not in ("log_price", "log_available_quantity")]
train_modelo = train[columnas_modelo]

carpeta_cache = tempfile.mkdtemp()
memoria = Memory(location=carpeta_cache, verbose=0)


def armar_pipeline(modelo):
    return Pipeline([
        ("capeo", CapeoPorMoneda(columnas=["price", "base_price"], columna_grupo="currency_id")),
        ("logs", FunctionTransformer(agregar_logs)),
        ("preprocesador", preprocesador_comparacion),
        ("modelo", modelo),
    ], memory=memoria)


modelos_ajustados = {}
filas_comparacion = []
accuracy_por_fold = {}


def ajustar_y_registrar(nombre, modelo, param_grid, n_jobs=4):
    """GridSearchCV con refit por accuracy. Guarda el mejor pipeline, sus métricas
    de CV y la accuracy de cada fold (para la comparación pareada)."""
    inicio = time.time()
    grid = GridSearchCV(armar_pipeline(modelo), param_grid, scoring=["accuracy", "roc_auc"],
                        refit="accuracy", cv=cv, n_jobs=n_jobs)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        grid.fit(train_modelo, y)

    mejor = grid.best_index_
    modelos_ajustados[nombre] = grid.best_estimator_
    accuracy_por_fold[nombre] = np.array([grid.cv_results_[f"split{k}_test_accuracy"][mejor] for k in range(5)])
    filas_comparacion.append({
        "modelo": nombre,
        "mejores_params": {k.replace("modelo__", ""): v for k, v in grid.best_params_.items()},
        "accuracy": grid.cv_results_["mean_test_accuracy"][mejor],
        "accuracy_std": grid.cv_results_["std_test_accuracy"][mejor],
        "roc_auc": grid.cv_results_["mean_test_roc_auc"][mejor],
    })
    print(nombre, "-> mejores params:", filas_comparacion[-1]["mejores_params"])
    print("   accuracy:", round(filas_comparacion[-1]["accuracy"], 4), "| roc_auc:", round(filas_comparacion[-1]["roc_auc"], 4),
          f"| {(time.time() - inicio) / 60:.1f} min", flush=True)
    return grid


def medir_ensamble(nombre, ensamble, scoring=("accuracy", "roc_auc"), mejores_params=None):
    """Mide un ensamble (sin grilla) con la misma CV y lo reentrena con todo train."""
    inicio = time.time()
    pipeline = armar_pipeline(ensamble)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        resultados = cross_validate(pipeline, train_modelo, y, cv=cv, scoring=list(scoring))
        pipeline.fit(train_modelo, y)

    modelos_ajustados[nombre] = pipeline
    accuracy_por_fold[nombre] = resultados["test_accuracy"]
    filas_comparacion.append({
        "modelo": nombre,
        "mejores_params": mejores_params,
        "accuracy": resultados["test_accuracy"].mean(),
        "accuracy_std": resultados["test_accuracy"].std(),
        "roc_auc": resultados["test_roc_auc"].mean() if "roc_auc" in scoring else np.nan,
    })
    print(nombre, "-> accuracy:", round(filas_comparacion[-1]["accuracy"], 4),
          "| roc_auc:", round(filas_comparacion[-1]["roc_auc"], 4), f"| {(time.time() - inicio) / 60:.1f} min", flush=True)

## Los 11 modelos

Mismas grillas que `train.ipynb`. Resumen de la justificación de cada una (detalle en `train.ipynb`, sección 3.3):

| Modelo | Hiperparámetros | Material |
|---|---|---|
| Regresión logística | `penalty` (L1/L2), `C` | `Continuación de regresion y Regulización L1 y L2.pdf` |
| LDA | `solver`, `shrinkage` | `Presentación_NB_LDA_QDA.pdf` |
| QDA | `reg_param` | `Presentación_NB_LDA_QDA.pdf` |
| Naive Bayes Gaussiano | `var_smoothing` | `Presentación_NB_LDA_QDA.pdf` |
| kNN | `n_neighbors`, `weights` | `Presentacion_kNN.pdf` |
| SVM lineal | `C` | `SVM (1).pdf` |
| Árbol de decisión | `criterion`, `max_depth`, `ccp_alpha` | `Árboles.pdf` |
| Random Forest | `n_estimators`, `max_depth` (`max_features="sqrt"` fijo) | `S8_Ensambles.ipynb` |
| Extra Trees | `n_estimators`, `max_depth` | `S8_Ensambles.ipynb` |
| AdaBoost | `n_estimators`, `learning_rate` | `S8_Ensambles.ipynb` |
| Gradient Boosting (`HistGradientBoostingClassifier`) | `max_iter`, `learning_rate`, `max_depth` | `S8_Ensambles.ipynb` |

`n_jobs` bajo en regresión logística y kNN por memoria, igual que en `train.ipynb`.

In [4]:
if correr("Regresión logística"):
    ajustar_y_registrar(
        "Regresión logística",
        LogisticRegression(max_iter=1000, random_state=SEED, solver="liblinear"),
        {"modelo__penalty": ["l1", "l2"], "modelo__C": [0.01, 0.1, 1, 10, 100]},
    )

Regresión logística -> mejores params: {'C': 1, 'penalty': 'l1'}
   accuracy: 0.9218 | roc_auc: 0.9765 | 39.9 min


In [5]:
if correr("LDA"):
    ajustar_y_registrar("LDA", LinearDiscriminantAnalysis(),
                        {"modelo__solver": ["lsqr", "eigen"], "modelo__shrinkage": [None, "auto", 0.1, 0.5, 0.9]})

LDA: salteado (False en MODELOS)


In [6]:
if correr("QDA"):
    ajustar_y_registrar("QDA", QuadraticDiscriminantAnalysis(),
                        {"modelo__reg_param": [0.0, 0.05, 0.1, 0.2, 0.3, 0.5]})

QDA: salteado (False en MODELOS)


In [7]:
if correr("Naive Bayes (Gaussiano)"):
    ajustar_y_registrar("Naive Bayes (Gaussiano)", GaussianNB(),
                        {"modelo__var_smoothing": np.logspace(-9, -1, 9)})

Naive Bayes (Gaussiano): salteado (False en MODELOS)


In [8]:
if correr("kNN"):
    ajustar_y_registrar("kNN", KNeighborsClassifier(),
                        {"modelo__n_neighbors": [5, 9, 15, 21, 31], "modelo__weights": ["uniform", "distance"]},
                        n_jobs=1)

kNN: salteado (False en MODELOS)


In [9]:
if correr("SVM lineal"):
    ajustar_y_registrar("SVM lineal", LinearSVC(max_iter=5000, random_state=SEED),
                        {"modelo__C": [0.01, 0.1, 1, 10]})

SVM lineal: salteado (False en MODELOS)


In [10]:
if correr("Árbol de decisión"):
    ajustar_y_registrar("Árbol de decisión", DecisionTreeClassifier(random_state=SEED),
                        {"modelo__criterion": ["gini", "entropy"], "modelo__max_depth": [3, 5, 10, None],
                         "modelo__ccp_alpha": [0.0, 0.001, 0.01]})

Árbol de decisión: salteado (False en MODELOS)


In [11]:
if correr("Random Forest"):
    ajustar_y_registrar("Random Forest", RandomForestClassifier(max_features="sqrt", random_state=SEED),
                        {"modelo__n_estimators": [100, 200], "modelo__max_depth": [10, 20]})

Random Forest -> mejores params: {'max_depth': 20, 'n_estimators': 200}
   accuracy: 0.9291 | roc_auc: 0.9787 | 1.7 min


In [12]:
if correr("Extra Trees"):
    ajustar_y_registrar("Extra Trees", ExtraTreesClassifier(max_features="sqrt", random_state=SEED),
                        {"modelo__n_estimators": [100, 200], "modelo__max_depth": [10, 20]})

Extra Trees: salteado (False en MODELOS)


In [13]:
if correr("AdaBoost"):
    ajustar_y_registrar("AdaBoost", AdaBoostClassifier(random_state=SEED),
                        {"modelo__n_estimators": [50, 100, 200], "modelo__learning_rate": [0.1, 0.5, 1.0]})

AdaBoost: salteado (False en MODELOS)


### Gradient Boosting

Misma grilla que `train.ipynb`. Incluye la configuración **fija** de los pasos 1 a 6 (`max_iter=200`, `learning_rate=0.1`, `max_depth=None`), que en `12_tuning` no se pudo mejorar. La guardo aparte como **"HistGB fijo"**, para la comparación pareada del final (sale de la misma grilla, no hace falta volver a entrenar).

In [14]:
if correr("Gradient Boosting"):
    grid_histgb = ajustar_y_registrar(
        "Gradient Boosting",
        HistGradientBoostingClassifier(random_state=SEED),
        {"modelo__max_iter": [100, 200], "modelo__learning_rate": [0.05, 0.1], "modelo__max_depth": [None, 5]},
    )

    resultados_histgb = pd.DataFrame(grid_histgb.cv_results_)
    fila_fijo = resultados_histgb[
        (resultados_histgb["param_modelo__max_iter"] == 200)
        & (resultados_histgb["param_modelo__learning_rate"] == 0.1)
        & (resultados_histgb["param_modelo__max_depth"].isna())
    ].index[0]
    accuracy_por_fold["HistGB fijo"] = np.array([resultados_histgb.loc[fila_fijo, f"split{k}_test_accuracy"] for k in range(5)])
    print("HistGB fijo -> accuracy:", round(accuracy_por_fold["HistGB fijo"].mean(), 4),
          "| roc_auc:", round(resultados_histgb.loc[fila_fijo, "mean_test_roc_auc"], 4))

Gradient Boosting -> mejores params: {'learning_rate': 0.1, 'max_depth': None, 'max_iter': 200}
   accuracy: 0.9384 | roc_auc: 0.9845 | 3.3 min


HistGB fijo -> accuracy: 0.9384 | roc_auc: 0.9845


## Ensambles

Misma lógica que `train.ipynb` (secciones 3.3.12 a 3.3.14), combinando los modelos ya tuneados de arriba que **se hayan corrido** (no se vuelve a tunear nada):

- **Voting (soft)**: promedia probabilidades. Excluye `SVM lineal` (sin `predict_proba`), `QDA` y `Naive Bayes` (muy por debajo del resto, arrastrarían el promedio).
- **Voting (hard)**: voto por mayoría. Suma `SVM lineal`. Sin `roc_auc` (no tiene `predict_proba`).
- **Stacking**: una regresión logística aprende a combinar las probabilidades *out-of-fold* (CV interna `cv=3`) de los **4 mejores modelos individuales por accuracy**. En `train.ipynb` el criterio era por `roc_auc`; acá es accuracy, coherente con `refit="accuracy"`.

CatBoost no entra en estos ensambles: necesita otro preprocesamiento (categóricas crudas) y no comparte el `Pipeline`. Combinarlo con los demás ya se evaluó en `16_ensambles.ipynb` (voting simple y ponderado, Hill Climber, Stacking), y ninguno lo mejoró.

In [15]:
if correr("Voting (soft)"):
    excluidos_voting = {"SVM lineal", "QDA", "Naive Bayes (Gaussiano)", "Voting (hard)", "Stacking"}
    estimadores_voting = [
        (nombre, modelos_ajustados[nombre].named_steps["modelo"])
        for nombre in list(modelos_ajustados)
        if nombre not in excluidos_voting
    ]
    print("Modelos en el Voting (soft):", [nombre for nombre, _ in estimadores_voting])

    assert len(estimadores_voting) >= 2, "El ensamble necesita al menos 2 modelos corridos"
    medir_ensamble("Voting (soft)", VotingClassifier(estimators=estimadores_voting, voting="soft"),
                   mejores_params={"estimators": [nombre for nombre, _ in estimadores_voting]})

Modelos en el Voting (soft): ['Regresión logística', 'Random Forest', 'Gradient Boosting']


Voting (soft) -> accuracy: 0.9357 | roc_auc: 0.9823 | 3.7 min


In [16]:
if correr("Voting (hard)"):
    excluidos_voting_hard = {"QDA", "Naive Bayes (Gaussiano)", "Voting (soft)", "Stacking"}
    estimadores_voting_hard = [
        (nombre, modelos_ajustados[nombre].named_steps["modelo"])
        for nombre in list(modelos_ajustados)
        if nombre not in excluidos_voting_hard
    ]
    print("Modelos en el Voting (hard):", [nombre for nombre, _ in estimadores_voting_hard])

    assert len(estimadores_voting_hard) >= 2, "El ensamble necesita al menos 2 modelos corridos"
    medir_ensamble("Voting (hard)", VotingClassifier(estimators=estimadores_voting_hard, voting="hard"),
                   scoring=("accuracy",),
                   mejores_params={"estimators": [nombre for nombre, _ in estimadores_voting_hard]})

Voting (hard): salteado (False en MODELOS)


In [17]:
if correr("Stacking"):
    individuales = pd.DataFrame(filas_comparacion).set_index("modelo")
    individuales = individuales.drop(index=["Voting (soft)", "Voting (hard)", "SVM lineal"], errors="ignore")
    modelos_para_stacking = individuales["accuracy"].sort_values(ascending=False).head(4).index.tolist()
    print("Modelos en el Stacking:", modelos_para_stacking)

    estimadores_stacking = [(nombre, modelos_ajustados[nombre].named_steps["modelo"]) for nombre in modelos_para_stacking]

    assert len(estimadores_stacking) >= 2, "El ensamble necesita al menos 2 modelos corridos"
    medir_ensamble(
        "Stacking",
        StackingClassifier(estimators=estimadores_stacking,
                           final_estimator=LogisticRegression(max_iter=1000, random_state=SEED), cv=3),
        mejores_params={"estimators": modelos_para_stacking, "final_estimator": "LogisticRegression", "cv_interno": 3},
    )

Modelos en el Stacking: ['Gradient Boosting', 'Random Forest', 'Regresión logística']


Stacking -> accuracy: 0.9384 | roc_auc: 0.9831 | 9.0 min


### El modelo final: CatBoost, HistGB con combinaciones y su promedio 50/50

Con `cross_val_predict(method="predict_proba")` (misma CV) se obtienen las probabilidades *out-of-fold* de CatBoost y del HistGB con combinaciones. Con ellas se calculan, fold por fold, la accuracy y la roc_auc de cada modelo y del promedio 50/50 (el modelo final). El promedio no tiene nada que elegir, así que no hace falta rotar. Material: `Árboles.pdf` (slide 68, CatBoost; slides 73–75, Voting) y `S8_Ensambles.ipynb` (voto *soft*).

In [18]:
if correr("Modelo final (CatBoost + HistGB con combinaciones + promedio)"):
    from sklearn.model_selection import cross_val_predict
    from sklearn.metrics import roc_auc_score

    inicio = time.time()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        prob_catboost = cross_val_predict(armar_pipeline_final(), train, y, cv=cv, method="predict_proba", n_jobs=5,
                                          params={"modelo__cat_features": INDICES_CAT})[:, 1]
        prob_histgb_combos = cross_val_predict(armar_pipeline_histgb_combos(), train, y, cv=cv, method="predict_proba",
                                               n_jobs=5)[:, 1]
    print(f"Probabilidades OOF: {(time.time() - inicio) / 60:.1f} min")

    folds = list(cv.split(train, y))
    for nombre, probabilidad, params in [
        ("CatBoost", prob_catboost, {"parámetros": "por defecto", "categóricas": "crudas"}),
        ("HistGB con combinaciones", prob_histgb_combos, {"codificación": "TargetEncoder sobre vendedor, categoría y combinaciones"}),
        ("Promedio 50/50 CatBoost + HistGB con combinaciones (modelo final)", 0.5 * prob_catboost + 0.5 * prob_histgb_combos,
         {"pesos": "50/50"}),
    ]:
        acc = np.array([((probabilidad[v] >= 0.5).astype(int) == y.iloc[v].to_numpy()).mean() for _, v in folds])
        auc = np.mean([roc_auc_score(y.iloc[v], probabilidad[v]) for _, v in folds])
        accuracy_por_fold[nombre] = acc
        filas_comparacion.append({"modelo": nombre, "mejores_params": params, "accuracy": acc.mean(),
                                  "accuracy_std": acc.std(), "roc_auc": auc})
        print(f"{nombre} -> accuracy: {acc.mean():.4f} | roc_auc: {auc:.4f}")

Probabilidades OOF: 3.0 min
CatBoost -> accuracy: 0.9414 | roc_auc: 0.9854
HistGB con combinaciones -> accuracy: 0.9412 | roc_auc: 0.9855
Promedio 50/50 CatBoost + HistGB con combinaciones (modelo final) -> accuracy: 0.9420 | roc_auc: 0.9860


### La entrega actual: + BETO (Paso 8)

El modelo de arriba es P7c + duración, **sin** BETO (`con_beto=False`). La entrega le cambia una sola cosa: `prob_texto_new` = (logística + BETO) / 2 (variante a de la Fase T, `29_texto_combinado`). Reentrenar BETO acá llevaría ~4,4 h, así que se usan las probabilidades *out-of-fold* **exactas** ya guardadas en la Fase T (`oof_texto_combinado.csv`). Son los mismos 5 folds (`StratifiedKFold(5, shuffle=True, random_state=42)`): el `assert` lo verifica.

In [19]:
if correr("Modelo final (CatBoost + HistGB con combinaciones + promedio)"):
    oof_t = pd.read_csv(".cache_laboratorio/oof_texto_combinado.csv")
    fold_de = np.empty(len(train), dtype=int)
    for k, (_, v) in enumerate(folds):
        fold_de[v] = k + 1
    assert (oof_t["fold"].to_numpy() == fold_de).all() and (oof_t["y"].to_numpy() == y.to_numpy()).all()
    p_cat_beto = oof_t["catboost (a) promedio simple"].to_numpy()
    p_hgb_beto = oof_t["(a) promedio simple"].to_numpy()
    for nombre, probabilidad in [("CatBoost + BETO", p_cat_beto), ("HistGB con combinaciones + BETO", p_hgb_beto),
                                 ("Entrega actual: promedio 50/50 + BETO", 0.5 * p_cat_beto + 0.5 * p_hgb_beto)]:
        acc = np.array([((probabilidad[v] >= 0.5).astype(int) == y.iloc[v].to_numpy()).mean() for _, v in folds])
        auc = np.mean([roc_auc_score(y.iloc[v], probabilidad[v]) for _, v in folds])
        accuracy_por_fold[nombre] = acc
        filas_comparacion.append({"modelo": nombre, "mejores_params": {"texto": "(logística + BETO) / 2", "origen": "OOF Fase T"},
                                  "accuracy": acc.mean(), "accuracy_std": acc.std(), "roc_auc": auc})
        print(f"{nombre}: accuracy {acc.mean():.4f} | roc_auc {auc:.4f}")

CatBoost + BETO: accuracy 0.9431 | roc_auc 0.9859
HistGB con combinaciones + BETO: accuracy 0.9425 | roc_auc 0.9859
Entrega actual: promedio 50/50 + BETO: accuracy 0.9435 | roc_auc 0.9864


`SVM lineal` queda afuera de los candidatos a Stacking porque `LinearSVC` no tiene `predict_proba` (el meta-modelo combina probabilidades).

## Tabla final

In [20]:
comparacion_final = pd.DataFrame(filas_comparacion).set_index("modelo").sort_values("accuracy", ascending=False)
comparacion_final[["accuracy", "accuracy_std", "roc_auc", "mejores_params"]].round(4)

,accuracy,accuracy_std,roc_auc,mejores_params
modelo,,,,
Entrega actual: promedio 50/50 + BETO,0.9435,0.0008,0.9864,"{'texto': '(logística + BETO) / 2', 'origen': ..."
CatBoost + BETO,0.9431,0.0007,0.9859,"{'texto': '(logística + BETO) / 2', 'origen': ..."
HistGB con combinaciones + BETO,0.9425,0.0014,0.9859,"{'texto': '(logística + BETO) / 2', 'origen': ..."
Promedio 50/50 CatBoost + HistGB con combinaciones (modelo final),0.9420,0.0017,0.9860,{'pesos': '50/50'}
CatBoost,0.9414,0.0013,0.9854,"{'parámetros': 'por defecto', 'categóricas': '..."
HistGB con combinaciones,0.9412,0.0016,0.9855,{'codificación': 'TargetEncoder sobre vendedor...
Stacking,0.9384,0.0017,0.9831,"{'estimators': ['Gradient Boosting', 'Random F..."
Gradient Boosting,0.9384,0.0014,0.9845,"{'learning_rate': 0.1, 'max_depth': None, 'max..."
Voting (soft),0.9357,0.0021,0.9823,"{'estimators': ['Regresión logística', 'Random..."


## El mejor contra el resto, fold por fold

Misma comparación pareada de los pasos 3 a 6: **mejora real** si gana en 5 de 5 folds, o en 4 de 5 con la media de la diferencia mayor que su desvío. Todos los modelos se midieron sobre los mismos 5 folds. Se compara el mejor de la tabla contra los siguientes 3 y contra el HistGB fijo de los pasos 1 a 6.

In [21]:
def comparar_pareado(nombre_a, nombre_b):
    """Diferencia de accuracy fold por fold (B - A), sobre los mismos folds."""
    diferencias = accuracy_por_fold[nombre_b] - accuracy_por_fold[nombre_a]
    print(f"{nombre_b}  vs  {nombre_a}")
    print("  diferencia por fold:", np.round(diferencias, 4))
    print("  folds en que gana B:", (diferencias > 0).sum(), "de", len(diferencias))
    print("  media de la diferencia:", round(diferencias.mean(), 4), "| desvío de la diferencia:", round(diferencias.std(), 4))


assert len(comparacion_final) >= 2, "Hacen falta al menos 2 modelos corridos para comparar"
mejor_modelo = comparacion_final.index[0]
for nombre in list(comparacion_final.index[1:4]) + ["HistGB fijo"]:
    if nombre in accuracy_por_fold and nombre != mejor_modelo:
        comparar_pareado(nombre, mejor_modelo)
        print()

Entrega actual: promedio 50/50 + BETO  vs  CatBoost + BETO
  diferencia por fold: [ 0.0009 -0.0003  0.0012 -0.0001  0.0003]
  folds en que gana B: 3 de 5
  media de la diferencia: 0.0004 | desvío de la diferencia: 0.0006

Entrega actual: promedio 50/50 + BETO  vs  HistGB con combinaciones + BETO
  diferencia por fold: [-0.0001  0.0022  0.      0.001   0.0019]
  folds en que gana B: 3 de 5
  media de la diferencia: 0.001 | desvío de la diferencia: 0.0009

Entrega actual: promedio 50/50 + BETO  vs  Promedio 50/50 CatBoost + HistGB con combinaciones (modelo final)
  diferencia por fold: [0.0001 0.0034 0.0019 0.0015 0.0009]
  folds en que gana B: 5 de 5
  media de la diferencia: 0.0015 | desvío de la diferencia: 0.0011

Entrega actual: promedio 50/50 + BETO  vs  HistGB fijo
  diferencia por fold: [0.0048 0.0064 0.0059 0.0036 0.0051]
  folds en que gana B: 5 de 5
  media de la diferencia: 0.0052 | desvío de la diferencia: 0.001



In [22]:
shutil.rmtree(carpeta_cache, ignore_errors=True)

## Conclusión

Corrida completa el 2026-11-09 (60,7 min, máquina libre): regresión logística, Random Forest, Gradient Boosting (HistGB), Voting (soft), Stacking y el modelo final. Misma StratifiedKFold(5, shuffle, 42) para todos.

| Modelo | Accuracy | Desvío | ROC AUC |
|---|---|---|---|
| **Entrega actual: promedio 50/50 + BETO** | **0,9435** | 0,0008 | 0,9864 |
| CatBoost + BETO | 0,9431 | 0,0007 | 0,9859 |
| HistGB con combinaciones + BETO | 0,9425 | 0,0014 | 0,9859 |
| Promedio 50/50 sin BETO (P7c + duración) | 0,9420 | 0,0017 | 0,9860 |
| CatBoost | 0,9414 | 0,0013 | 0,9854 |
| HistGB con combinaciones | 0,9412 | 0,0016 | 0,9855 |
| Stacking (GB + RF + logística) | 0,9384 | 0,0017 | 0,9831 |
| Gradient Boosting (HistGB, grilla) | 0,9384 | 0,0014 | 0,9845 |
| Voting soft (logística + RF + GB) | 0,9357 | 0,0021 | 0,9823 |
| Random Forest (`max_depth=20`, 200 árboles) | 0,9291 | 0,0017 | 0,9787 |
| Regresión logística (`C=1`, L1) | 0,9218 | 0,0021 | 0,9765 |

- **Familias:** con las mismas features, el boosting (0,9384) le gana al bagging (Random Forest, 0,9291) y al modelo lineal (0,9218). Combinar modelos más flojos no suma: el Voting empeora (0,9357) y el Stacking empata con el Gradient Boosting solo.
- **Lo que sí suma:** cómo se codifican el vendedor y la categoría (CatBoost con categóricas crudas y HistGB con target encoding de combinaciones, ~0,941 cada uno), promediar esos dos modelos que se equivocan distinto (0,9420) y la probabilidad de texto (logística + BETO)/2 (0,9435).
- **Comparación pareada de la entrega:** contra el promedio sin BETO, +0,0015 y gana 5/5 (pasa); contra el Gradient Boosting de la grilla, +0,0052 y 5/5. Contra CatBoost + BETO solo (+0,0004, 3/5) y HistGB + BETO solo (+0,0010, 3/5) no gana de forma clara: el promedio 50/50 se mantiene porque baja el desvío entre folds (0,0008 contra 0,0014 del HistGB) y se eligió antes con el mismo criterio (Fase F).
- Las filas "+ BETO" usan las probabilidades OOF guardadas de la Fase T (`oof_texto_combinado.csv`, mismos folds, verificado con `assert`), no se reentrenó BETO en esta notebook.